<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [3]</a>'.</span>

In [1]:
# Parameters
p_batch_id = "2025-01"


In [2]:
# ============================================================
# IDENTIFY NEXT BATCH — LOCAL EXECUTION
# ============================================================

import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F
import nbformat
from nbconvert import PythonExporter

# Load environment
def run_nb(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    src, _ = PythonExporter().from_notebook_node(nb)
    exec(src, globals())

run_nb("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")

control_file = f"{CONTROL_PATH}/batch_control/data.parquet"

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/10 07:48:37 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/10 07:48:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/10 07:48:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [3]:
# -----------------------------------------
# 1. Receive batch_id
# -----------------------------------------
p_batch_id = next_batch   # or manually set

if not p_batch_id:
    raise Exception("❌ No batch_id provided")

NameError: name 'next_batch' is not defined

In [ ]:
# -----------------------------------------
# 2. Read control table
# -----------------------------------------
df = spark.read.parquet(control_file)

In [ ]:
# -----------------------------------------
# 3. Update status
# -----------------------------------------
df_updated = (
    df.withColumn(
        "status",
        F.when(
            (F.col("batch_id") == p_batch_id) & (F.col("status") == "in_progress"),
            F.lit("completed")
        ).otherwise(F.col("status"))
    )
    .withColumn(
        "updated_timestamp",
        F.when(F.col("batch_id") == p_batch_id, F.current_timestamp())
         .otherwise(F.col("updated_timestamp"))
    )
)

In [ ]:
# -----------------------------------------
# 4. Write back
# -----------------------------------------
df_updated.write.mode("overwrite").parquet(control_file)

print(f"✔ Marked batch {p_batch_id} as completed")